# Predição de resultados do Brasileirão com Naive Bayes

Notebook convertido a partir do arquivo Python original para treinar e avaliar um modelo Naive Bayes Gaussiano.

## 1. Importações e configurações

Define bibliotecas, caminho do dataset, ano de teste, coluna alvo e listas de atributos usados pelo modelo.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATASET_PATH = Path("brasileirao_serie_a_2018_2023_v3.csv")
TEST_YEAR = 2023
TARGET_COLUMN = "vencedor"

NUMERIC_FEATURES = [
    "ano_campeonato",
    "mes_campeonato",
    "rodada",
    "PPJ_pre_jogo_mandante",
    "PPJ_pre_jogo_visitante",
    "xG_pre_jogo_mandante",
    "xG_pre_jogo_visitante",
    "GPJ_pre_jogo",
    "AM_porcentagem_pre_jogo",
    "A15_porcentagem_pre_jogo",
    "A25_porcentagem_pre_jogo",
    "A45_porcentagem_pre_jogo",
    "EPJ_pre_jogo",
    "odds_mandante_vence",
    "odds_empate",
    "odds_visitante_vence",
    "colocacao_mandante",
    "colocacao_visitante",
    "valor_equipe_titular_mandante",
    "valor_equipe_titular_visitante",
    "idade_media_titular_mandante",
    "idade_media_titular_visitante",
]

CATEGORICAL_FEATURES = [
    "time_mandante",
    "time_visitante",
    "estadio",
    "mandante_estado",
    "visitante_estado",
    "formacao_mandante",
    "formacao_visitante",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

## 2. Construção do modelo

Cria o pipeline de pré-processamento e classificação com `GaussianNB`.

In [2]:
def build_model() -> Pipeline:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, NUMERIC_FEATURES),
            ("cat", categorical_transformer, CATEGORICAL_FEATURES),
        ],
        sparse_threshold=0.0,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", GaussianNB()),
        ]
    )

## 3. Carregamento do dataset

Lê o CSV e valida se todas as colunas necessárias estão presentes.

In [3]:
def load_dataset() -> pd.DataFrame:
    if not DATASET_PATH.exists():
        raise FileNotFoundError(f"Dataset não encontrado: {DATASET_PATH}")

    df = pd.read_csv(DATASET_PATH)
    missing_columns = [
        column for column in [*FEATURES, TARGET_COLUMN] if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"Colunas ausentes no CSV: {missing_columns}")

    return df

## 4. Preparação dos dados

Remove empates e separa treino e teste por ano do campeonato.

In [4]:
def prepare_data(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series, int]:
    draws_count = int((df[TARGET_COLUMN] == "empate").sum())
    df_without_draws = df[df[TARGET_COLUMN] != "empate"].copy()

    train_df = df_without_draws[df_without_draws["ano_campeonato"] < TEST_YEAR]
    test_df = df_without_draws[df_without_draws["ano_campeonato"] == TEST_YEAR]

    if train_df.empty:
        raise ValueError(f"Nenhum jogo encontrado para treino antes de {TEST_YEAR}.")

    if test_df.empty:
        raise ValueError(f"Nenhum jogo encontrado para teste em {TEST_YEAR}.")

    X_train = train_df[FEATURES]
    y_train = train_df[TARGET_COLUMN]
    X_test = test_df[FEATURES]
    y_test = test_df[TARGET_COLUMN]

    return X_train, y_train, X_test, y_test, draws_count

## 5. Exemplos de predição

Monta uma tabela com os primeiros jogos do teste, resultado real, previsão e probabilidades por classe.

In [5]:
def show_prediction_examples(
    model: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    predictions: list[str],
    examples_count: int = 10,
) -> pd.DataFrame:
    probabilities = model.predict_proba(X_test)
    classes = list(model.classes_)

    results = X_test[
        ["ano_campeonato", "rodada", "time_mandante", "time_visitante"]
    ].copy()
    results["vencedor_real"] = y_test.values
    results["vencedor_previsto"] = predictions

    for index, class_name in enumerate(classes):
        results[f"prob_{class_name}"] = probabilities[:, index]

    return results.head(examples_count)

## 6. Treinamento e avaliação

Executa o fluxo principal: carrega os dados, treina o modelo, prediz os resultados do teste e calcula métricas.

In [6]:
df = load_dataset()
X_train, y_train, X_test, y_test, draws_count = prepare_data(df)

model = build_model()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Modelo: Naive Bayes Gaussiano")
print(f"Dataset: {DATASET_PATH.name}")
print(f"Empates removidos: {draws_count}")
print(f"Exemplos de treino: {len(X_train)}")
print(f"Exemplos de teste ({TEST_YEAR}): {len(X_test)}")
print(f"Classes: {list(model.classes_)}")

accuracy = accuracy_score(y_test, predictions)
print(f"\nAcurácia no teste: {accuracy:.4f}")

Modelo: Naive Bayes Gaussiano
Dataset: brasileirao_serie_a_2018_2023_v3.csv
Empates removidos: 574
Exemplos de treino: 1259
Exemplos de teste (2023): 213
Classes: [np.str_('casa'), np.str_('fora')]

Acurácia no teste: 0.6526


## 7. Relatório de classificação

In [7]:
print(classification_report(y_test, predictions, digits=4))

              precision    recall  f1-score   support

        casa     0.7037    0.8143    0.7550       140
        fora     0.4902    0.3425    0.4032        73

    accuracy                         0.6526       213
   macro avg     0.5969    0.5784    0.5791       213
weighted avg     0.6305    0.6526    0.6344       213



## 8. Matriz de confusão

In [8]:
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=model.classes_),
    index=model.classes_,
    columns=model.classes_,
)

confusion_matrix_df

,casa,fora
casa,114,26
fora,48,25


## 9. Exemplos de predições

In [9]:
prediction_examples = show_prediction_examples(model, X_test, y_test, predictions)
prediction_examples

,ano_campeonato,rodada,time_mandante,time_visitante,vencedor_real,vencedor_previsto,prob_casa,prob_fora
429,2023,9,Vasco da Gama,Flamengo,fora,fora,0.003524,0.996476
702,2023,20,Santos,Grêmio,casa,casa,0.580678,0.419322
1549,2023,1,Palmeiras,Cuiabá,casa,casa,0.987025,0.012975
1551,2023,2,Fluminense,Athletico-PR,casa,casa,0.750896,0.249104
1552,2023,2,Cruzeiro,Grêmio,casa,casa,0.660004,0.339996
1554,2023,1,Flamengo,Coritiba,casa,casa,0.979471,0.020529
1557,2023,1,Botafogo,São Paulo,casa,casa,0.641266,0.358734
1560,2023,1,América-MG,Fluminense,fora,fora,0.498724,0.501276
1562,2023,2,Internacional,Flamengo,casa,fora,0.002025,0.997975
1565,2023,1,Bragantino,Bahia,casa,casa,0.721728,0.278272
